# [1장 2강] - 실습: 내적과 코사인 유사도
**실습 목표**

- 고객 구매 이력을 벡터로 표현하고 두 벡터의 내적을 직접 계산할 수 있다.
- 코사인 유사도를 `cosθ = a·b / (|a||b|)` 공식으로 구현하고 값의 범위를 해석할 수 있다.
- 내적과 코사인 유사도의 차이를 구매량 규모 관점에서 설명할 수 있다.
- 유사도 검색으로 유사 고객과 추천 후보를 찾는 과정을 구현할 수 있다.

**진행 방식**

- 사용 도구: Google Colab 또는 Jupyter Notebook, Python, NumPy, pandas, scikit-learn
- 고객-상품 행렬 생성 → 내적 계산 → 코사인 유사도 구현 → 유사도 검색 순서로 진행합니다.
- 개인 실습으로 진행하며, 계산 결과를 "고객 관점의 해석 문장"으로 바꿔 적어 봅니다.

**실습에 필요한 데이터셋/파일**

- 분야: 이커머스
- 데이터셋: UCI Online Retail
- 사용 방식: `ucimlrepo.fetch_ucirepo(id=352)`
- 출처: https://archive.ics.uci.edu/dataset/352/online+retail
- 사용 목적: `CustomerID`, `StockCode`, `Quantity` 컬럼으로 고객-상품 구매량 행렬을 만들어 내적·코사인 유사도를 계산합니다.
- 준비물: Python, NumPy, pandas, scikit-learn, ucimlrepo

**데이터 로드 시 유의사항**

- `ucimlrepo`는 Colab에 기본 설치되어 있지 않습니다. 아래 준비 코드의 `!pip install -q ucimlrepo` **셀을 반드시 먼저 실행**하세요. 설치를 건너뛰면 실제 Online Retail이 아니라 임의로 생성한 대체 데이터로 실습이 진행되며, 이 사실을 모른 채 결과를 해석하기 쉽습니다.
- `fetch_ucirepo()`는 UCI 서버에 네트워크로 접속합니다. 사내망·방화벽·서버 점검 등으로 **로드가 실패할 수 있으므로** 예외 처리를 두고, 실패 시 같은 컬럼 구조의 대체 거래 데이터로 실습을 이어갑니다.
- Online Retail에는 `CustomerID`가 비어 있는 거래가 많습니다. 고객 단위 분석이므로 **결측 고객 행은 먼저 제거**합니다.
- 전체 상품은 수천 종이라 행렬이 지나치게 커집니다. 구매량 상위 고객 60명 × 상위 상품 150개로 잘라 실습 규모를 맞춥니다. 상품 수를 너무 적게(50개 이하) 잡으면 심화 문제에서 추천 후보가 거의 남지 않으므로 150개 이상을 사용합니다.

In [2]:
# 최초 1회만 실행 (새 환경일 때)
# Colab에는 ucimlrepo가 기본 설치되어 있지 않으므로 이 셀을 건너뛰지 마세요.

# !는 뒤에 오는 걸 파이썬이 아니라 셸(터미널) 명령어로 실행하라는 뜻
# %는 셸로 넘기는 게 아니라, IPython(주피터 커널)이 자체적으로 제공하는 특수 명령
# 지금 이 커널이 쓰는 바로 그 파이썬에 설치하도록 IPython이 알아서 설치
%pip install -q ucimlrepo

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


def load_retail():
    """Online Retail 거래 데이터를 불러오고, 실패하면 같은 구조의 대체 데이터를 사용합니다."""
    try:
        from ucimlrepo import fetch_ucirepo
    except ImportError:
        print('[경고] ucimlrepo 패키지가 설치되어 있지 않습니다. 위의 pip 설치 셀을 먼저 실행하세요.')
        print('[경고] 지금은 실제 데이터가 아닌 대체(임의 생성) 거래 데이터로 진행됩니다.')
        return _dummy_retail()

    try:
        ds = fetch_ucirepo(id=352)
        return ds.data.original.copy()
    except Exception as e:
        print('[경고] UCI 서버 접속 실패(네트워크·방화벽 확인 필요):', e)
        print('[경고] 동일한 컬럼 구조의 대체 거래 데이터로 진행합니다. 출력값은 교안 예시와 다릅니다.')
        return _dummy_retail()


def _dummy_retail():
    """UCI를 쓸 수 없을 때 사용하는 같은 컬럼 구조의 대체 거래 데이터입니다."""
    rng = np.random.default_rng(RANDOM_STATE)
    n = 6000
    return pd.DataFrame({
        'InvoiceNo': rng.integers(10000, 10800, n).astype(str),
        'StockCode': rng.choice([f'P{i:03d}' for i in range(300)], n),
        'Quantity': rng.poisson(3, n) + 1,
        'UnitPrice': rng.gamma(2.0, 10.0, n),
        'CustomerID': rng.integers(1000, 1120, n),
    })


def customer_product_matrix(df, n_customers=60, n_products=150):
    """고객(행) x 상품(열) 구매량 행렬을 만듭니다."""
    df = df.dropna(subset=['CustomerID', 'StockCode']).copy()  # 고객 미상 거래 제외
    pivot = df.pivot_table(index='CustomerID', columns='StockCode',
                           values='Quantity', aggfunc='sum', fill_value=0)
    pivot = pivot.loc[pivot.sum(axis=1).sort_values(ascending=False).head(n_customers).index]
    pivot = pivot[pivot.sum(axis=0).sort_values(ascending=False).head(n_products).index]
    return pivot.astype(float)


retail = load_retail()
M_df = customer_product_matrix(retail, 60, 150)
M = M_df.values
print('고객-상품 행렬:', M.shape)

Note: you may need to restart the kernel to use updated packages.
고객-상품 행렬: (60, 150)


## 필수 1 : 고객 구매 이력을 벡터로 만들고 겹치는 정도 재기
이커머스 CRM팀은 "이 고객과 취향이 비슷한 고객"을 찾아 교차 추천 대상을 만들려고 합니다. 그러려면 먼저 고객 한 명의 구매 이력을 상품별 구매량 벡터로 바꾸고, 두 고객이 같은 상품을 얼마나 함께 샀는지를 숫자 하나로 요약해야 합니다. 이 계산이 바로 내적입니다.

### 문제 1-1 : 고객 벡터의 내적 직접 계산하기
1. `M_df`에서 0번, 1번 고객의 구매량 벡터 `a`, `b`를 추출하고 shape을 확인합니다.
2. 두 벡터의 내적을 `np.sum(a * b)`로 직접 계산합니다.
3. `np.dot(a, b)`, `a @ b` 결과와 값이 같은지 확인합니다.
4. 내적 결과의 타입(스칼라)을 확인하고, 스칼라로 나오는 것이 왜 유용한지 한 문장으로 설명합니다.

In [3]:
# 1. `M_df`에서 0번, 1번 고객의 구매량 벡터 `a`, `b`를 추출하고 shape을 확인합니다.
a = M_df.iloc[0].values
print(f"shape: {a.shape}")
b = M_df.iloc[1].to_numpy()
print(f"shape: {b.shape}\n")

# 2. 두 벡터의 내적을 `np.sum(a * b)`로 직접 계산합니다.
inner_product = np.sum(a*b)
dot_numpy = np.dot(a, b)
print(inner_product, "\n")

# 3. `np.dot(a, b)`, `a @ b` 결과와 값이 같은지 확인합니다.
print(f"np.dot: {np.dot(a, b)}, a @ b: {a @ b}\n")

# 4. 내적 결과의 타입(스칼라)을 확인하고, 스칼라로 나오는 것이 왜 유용한지 한 문장으로 설명합니다.
print(type(inner_product), type(dot_numpy))
print(np.ndim(dot_numpy))


shape: (150,)
shape: (150,)

31537526.0 

np.dot: 31537526.0, a @ b: 31537526.0

<class 'numpy.float64'> <class 'numpy.float64'>
0


### 문제 1-2 : 내적 값이 구매 규모에 영향을 받는지 확인하기
1. 0번 고객과 나머지 모든 고객의 내적을 `M @ a`로 한 번에 계산합니다.
2. 내적을 내림차순으로 정렬해 **1위가 누구인지 먼저 확인**합니다. 그리고 **왜 1위가 항상 자기 자신인지**를 내적 공식(`a·a = |a|²`) 관점에서 한 문장으로 설명합니다. 이 때문에 이후 순위를 볼 때는 자기 자신을 제외해야 합니다.
3. 자기 자신을 제외한 내적 상위 5명을 찾고, 그 고객들의 **총 구매량**(행 합계)도 함께 출력합니다.
4. 전체 고객의 총 구매량 평균과 비교해, 상위 5명이 특별히 구매량이 많은 고객인지 확인합니다.
5. 내적만으로 "취향이 비슷한 고객"을 판단할 때 생길 수 있는 문제를 한 문장으로 작성합니다.

In [4]:
# 1. 0번 고객과 나머지 모든 고객의 내적을 `M @ a`로 한 번에 계산합니다.
# print(M_df.shape)     # (60, 150)
print(M_df.head(5))
# print(a.shape)        # (150,)

M_a = M_df @ a
# print(M_a.shape)      # (60,)
# print(M_a)
# print(type(M_a))      # <class 'pandas.Series'>

# 2. 내적을 내림차순으로 정렬해 1위가 누구인지 먼저 확인합니다. 그리고 왜 1위가 항상 자기 자신인지를 내적 공식(`a·a = |a|²`) 관점에서 한 문장으로 설명합니다. 이 때문에 이후 순위를 볼 때는 자기 자신을 제외해야 합니다.
ranking = M_a.sort_values(ascending=False)
# ranking.head(5)

# 3. 자기 자신을 제외한 내적 상위 5명을 찾고, 그 고객들의 **총 구매량**(행 합계)도 함께 출력합니다.
ranking_except = ranking[1:6]
top5_ids = ranking_except.index

total_qty = M_df.sum(axis=1)
top5_qty = total_qty.loc[top5_ids]

result = pd.DataFrame({
    '내적': ranking_except,
    '총구매량': top5_qty
})
# result
# 4. 전체 고객의 총 구매량 평균과 비교해, 상위 5명이 특별히 구매량이 많은 고객인지 확인합니다.
avg = total_qty.mean()
print(avg)

result2 = pd.DataFrame({
    '내적': ranking_except,
    '총구매량': top5_qty,
    '전체 고객 평균': avg
})
result2

# 5. 내적만으로 "취향이 비슷한 고객"을 판단할 때 생길 수 있는 문제를 한 문장으로 작성합니다.

StockCode   22197  84077  85099B  22616   21212  85123A   22492  21977  84826  \
CustomerID                                                                      
14646.0       0.0    0.0  2000.0    0.0  4104.0   416.0  1728.0  264.0    0.0   
12415.0       0.0    0.0   200.0    0.0   360.0     0.0  2880.0  240.0    0.0   
14911.0     921.0  528.0   110.0    0.0   672.0   746.0   252.0  240.0    0.0   
17450.0       0.0    0.0     0.0    0.0     0.0  4114.0     0.0    0.0    0.0   
18102.0       0.0    0.0   100.0    0.0     0.0     0.0     0.0    0.0    0.0   

StockCode    21915  ...  21981  22608  23308   23204  82580  22910   22328  \
CustomerID          ...                                                      
14646.0        0.0  ...  157.0    0.0    0.0  1301.0    0.0    0.0  1584.0   
12415.0     1680.0  ...  432.0    0.0  240.0     0.0    0.0   80.0     0.0   
14911.0      198.0  ...   72.0  101.0  240.0   100.0   24.0  362.0    12.0   
17450.0        0.0  ...    0.0    0.0    0

,내적,총구매량,전체 고객 평균
CustomerID,,,
12415.0,31537526.0,23696.0,12354.783333
13027.0,26962560.0,17280.0,12354.783333
15769.0,19886500.0,25600.0,12354.783333
17404.0,15825656.0,19004.0,12354.783333
14156.0,15345543.0,21271.0,12354.783333


## 필수 2 : 구매 규모를 제외하고 "구매 성향"만 비교하기

대량 구매 고객이 항상 유사 고객 상위에 오르면 추천이 한쪽으로 쏠립니다. CRM팀이 원하는 것은 "얼마나 많이 샀는가"가 아니라 "무엇을 사는 성향인가"입니다. 내적을 두 벡터 크기의 곱으로 나누어 방향만 비교하는 코사인 유사도를 계산합니다.

### 문제 2-1 : 코사인 유사도를 공식으로 직접 구현하기
1. `cosine(a, b) = np.dot(a, b) / (norm(a) * norm(b))` 함수를 직접 정의합니다.
2. 0번과 1번 고객의 코사인 유사도를 계산합니다.
3. 검증용으로 다음 세 경우의 코사인 유사도를 계산합니다.
    - 자기 자신과의 유사도
    - 벡터에 스칼라 3을 곱한 벡터와의 유사도
    - 서로 겹치는 상품이 없는(내적이 0인) 두 벡터의 유사도
4. 코사인 유사도 값 1 / 0 / -1이 각각 무엇을 뜻하는지 정리합니다.

In [ ]:
# 1. `cosine(a, b) = np.dot(a, b) / (norm(a) * norm(b))` 함수를 직접 정의합니다.
def cosine(a, b):
    result = (a @ b) / (np.linalg.norm(a) * np.linalg.norm(b))
    return result

# 2. 0번과 1번 고객의 코사인 유사도를 계산합니다.
cos_sim = cosine(a, b)
# print(cos_sim)

# 3. 검증용으로 다음 세 경우의 코사인 유사도를 계산합니다.
#     - 자기 자신과의 유사도
#     - 벡터에 스칼라 3을 곱한 벡터와의 유사도
#     - 서로 겹치는 상품이 없는(내적이 0인) 두 벡터의 유사도
test1 = cosine(a, a)
# print(test1)
test2 = cosine(a, 3* a)
# print(test2)

zero_ids = M_a[M_a == 0].index
zero_sample_v = M_df.loc[13256.0].values
test3 = cosine(a, zero_sample_v)
test3

# 4. 코사인 유사도 값 1 / 0 / -1이 각각 무엇을 뜻하는지 정리합니다.
# 코사인은 빗변 / 밑변 이다. 즉 빗변을 밑변으로 나눈다. 이때 코사인의 값이 1에 가까이 갈 수록 두 벡터의 방향이 같다는 의미이고,
# 0 값에 가까워질 수록 두 벡터의 방향이 유사하지 않다는 의미이다. -1일 경우 방향이 반대이다.

np.float64(0.0)

### 문제 2-2 : 유사 고객 상위 5명 찾기
1. `cosine_similarity(M)`으로 전체 고객 간 유사도 행렬을 계산하고 shape을 확인합니다.
2. 유사도 행렬을 고객 ID를 인덱스로 하는 DataFrame으로 만듭니다.
3. 0번 고객 기준 유사도 상위 5명을 출력합니다. (자기 자신 제외)
4. 문제 1-2의 내적 상위 5명과 목록이 어떻게 다른지 비교합니다.
5. 규모가 다른 고객을 비교할 때 코사인 유사도가 더 적합한 이유를 한 문장으로 작성합니다.

In [ ]:
# 1. `cosine_similarity(M)`으로 전체 고객 간 유사도 행렬을 계산하고 shape을 확인합니다.
import sklearn.metrics.pairwise
cos_sim = sklearn.metrics.pairwise.cosine_similarity(M_df)
print(cos_sim.shape)

# 2. 유사도 행렬을 고객 ID를 인덱스로 하는 DataFrame으로 만듭니다.
sim_df = pd.DataFrame(cos_sim, index=M_df.index, columns=M_df.index)
sim_df.head(5)

# 3. 0번 고객 기준 유사도 상위 5명을 출력합니다. (자기 자신 제외)
result = sim_df.iloc[0].sort_values(ascending=False)[1:6]
type(result)
print(result)

# 4. 문제 1-2의 내적 상위 5명과 목록이 어떻게 다른지 비교합니다.
result2
# 내적값이 크다고 해서 유사한 건 아니였다!

# 5. 규모가 다른 고객을 비교할 때 코사인 유사도가 더 적합한 이유를 한 문장으로 작성합니다.


(60, 60)
CustomerID
12415.0    0.549550
12678.0    0.496280
17428.0    0.484370
17735.0    0.476415
17677.0    0.383925
Name: 14646.0, dtype: float64


,내적,총구매량,전체 고객 평균
CustomerID,,,
12415.0,31537526.0,23696.0,12354.783333
13027.0,26962560.0,17280.0,12354.783333
15769.0,19886500.0,25600.0,12354.783333
17404.0,15825656.0,19004.0,12354.783333
14156.0,15345543.0,21271.0,12354.783333


## 심화 1 : 유사 고객을 이용한 추천 후보 만들기

검색 엔진이 질문 벡터와 가장 유사한 문서를 찾듯이, 추천 시스템은 기준 고객과 가장 유사한 고객을 찾은 뒤 그 고객이 산 상품 중 아직 사지 않은 것을 후보로 제시합니다. 앞에서 계산한 유사도를 실제 추천 목록으로 이어 봅니다.

### 문제 3-1 : 유사 고객 기반 추천 후보 도출하기
1. 0번 고객과 코사인 유사도가 가장 높은 이웃 고객 1명을 선택합니다.
2. 이웃 고객의 구매량 벡터를 추천 점수로 사용합니다.
3. 기준 고객이 **이미 구매한 상품은 후보에서 제외**합니다.
4. 이웃 고객도 구매하지 않은 상품(점수 0)은 추천 근거가 없으므로 **점수가 0보다 큰 상품만 후보로 남깁니다.** 남은 후보가 몇 개인지 먼저 출력한 뒤, 그중 점수 상위 10개(후보가 10개 미만이면 전부)를 출력합니다.
5. 이 방식이 임베딩 기반 유사도 검색과 어떤 점에서 같은 원리인지 한 문장으로 작성합니다.

In [43]:
# 1. 0번 고객과 코사인 유사도가 가장 높은 이웃 고객 1명을 선택합니다.
result
scores = M_df.loc[result.index[0]]
scores

# 2. 이웃 고객의 구매량 벡터를 추천 점수로 사용합니다.
base = M_df.iloc[0]

# 3. 기준 고객이 **이미 구매한 상품은 후보에서 제외**합니다.
candidates = scores[base == 0]

# 4. 이웃 고객도 구매하지 않은 상품(점수 0)은 추천 근거가 없으므로 **점수가 0보다 큰 상품만 후보로 남깁니다.** 남은 후보가 몇 개인지 먼저 출력한 뒤, 그중 점수 상위 10개(후보가 10개 미만이면 전부)를 출력합니다.
recommend = candidates[candidates != 0]
recommend
top10_recommend = recommend.sort_values(ascending=False)[:10]
top10_recommend

# 5. 이 방식이 임베딩 기반 유사도 검색과 어떤 점에서 같은 원리인지 한 문장으로 작성합니다.

StockCode
21915    1680.0
22969     970.0
15036     600.0
22722     576.0
21891     432.0
23308     240.0
23202     100.0
22379     100.0
22910      80.0
Name: 12415.0, dtype: float64